In [0]:
df = spark.read.table("workspace.auto_loader.demo_table2")
display(df)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import lit, rand, col, max as spark_max
import random

# Determine the max existing id so we can generate unique, non-duplicate primary keys
id_col = "id"
existing_max_id = df.agg(spark_max(col(id_col).cast("int"))).collect()[0][0]
existing_max_id = existing_max_id if existing_max_id is not None else 0

# Collect existing distinct values for text columns so we can distribute real data
def collect_distinct(col_name):
    if col_name not in df.columns:
        return []
    return [r[0] for r in df.select(col(col_name)).distinct().collect() if r[0] is not None]

products = collect_distinct("product")
remarks_vals = collect_distinct("remarks")
order_statuses = collect_distinct("order_status")

# Build 100 random records with unique ids, different names, and null for any column without real data
# Determine which columns are numeric so we can populate them with random values
numeric_types = ("int", "bigint", "double", "float", "long", "short")
numeric_cols = [c for c in df.columns if c != "_rescued_data" and c != id_col and dict(df.dtypes)[c] in numeric_types]
max_vals = {c: df.agg({c: "max"}).collect()[0][0] for c in numeric_cols}

# Detect the customer name column (if present) so each record gets a unique name
name_col = None
for candidate in ("customer_name", "name", "customerName"):
    if candidate in df.columns:
        name_col = candidate
        break

random_names = [
    "Alice Johnson", "Bob Smith", "Charlie Brown", "Diana Prince", "Edward Norton",
    "Fiona Apple", "George Lucas", "Hannah Montana", "Ian McKellen", "Julia Roberts",
    "Kevin Hart", "Laura Linney", "Michael Caine", "Natalie Portman", "Oscar Wilde",
    "Penelope Cruz", "Quentin Tarantino", "Rachel Green", "Samuel Jackson", "Tina Fey",
    "Uma Thurman", "Vin Diesel", "Will Smith", "Xavier Dolan", "Yvonne Strahovski",
    "Zachary Quinto", "Aaron Eckhart", "Bryan Cranston", "Catherine Zeta", "David Beckham",
    "Emma Stone", "Frank Sinatra", "Greta Garbo", "Harrison Ford", "Isla Fisher",
    "John Travolta", "Kate Winslet", "Leonardo DiCaprio", "Meryl Streep", "Nicolas Cage",
    "Olivia Wilde", "Pierce Brosnan", "Queen Latifah", "Robert Downey", "Sandra Bullock",
    "Tom Hanks", "Ursula Andress", "Viggo Mortensen", "Whoopi Goldberg", "Xander Berkeley",
    "Yara Shahidi", "Zoe Saldana", "Amanda Seyfried", "Bradley Cooper", "Cillian Murphy",
    "Drew Barrymore", "Ethan Hawke", "Florence Pugh", "Gwen Stefani", "Henry Cavill",
    "Iris Elba", "James Franco", "Kirsten Dunst", "Liam Neeson", "Mila Kunis",
    "Noah Centineo", "Olivia Coleman", "Patrick Swayze", "Quincy Jones", "Renee Zellweger",
    "Sebastian Stan", "Tilda Swinton", "Ulysses Grant", "Val Kilmer", "Winona Ryder",
    "Xenia Goodwin", "Yael Grobglas", "Zane Holtz", "Adrian Brody", "Bella Hadid",
    "Cara Delevingne", "Denzel Washington", "Eva Green", "Felicity Jones", "Gael Garcia",
    "Hailey Bieber", "Idris Elba", "Jennifer Lawrence", "Kit Harington", "Lupita Nyong",
    "Matthew McConaughey", "Naomi Watts", "Orlando Bloom", "Paul Rudd", "Rosamund Pike",
    "Sophia Loren", "Toby Maguire", "Uma Karuna", "Vince Vaughn", "Will Ferrell"
]

# Pre-select 10 random indices (out of 100) that will get NULL for each nullable text column
null_indices_name = set(random.sample(range(100), 17)) if name_col else set()
null_indices_product = set(random.sample(range(100), 10)) if products else set(range(100))
null_indices_remarks = set(random.sample(range(100), 10)) if remarks_vals else set(range(100))
null_indices_order = set(random.sample(range(100), 10)) if order_statuses else set(range(100))

# Special characters to inject into remarks values
special_chars = ["@", "#", "$", "%", "&", "*", "!", "~", "^", "+"]

def inject_special_chars(text):
    """Insert special characters at random positions in the text."""
    if not text:
        return text
    text = str(text)
    chars = list(text)
    # Insert 1-3 special characters at random positions
    for _ in range(random.randint(1, 3)):
        pos = random.randint(0, len(chars))
        chars.insert(pos, random.choice(special_chars))
    return "".join(chars)

random_rows = []
for i in range(100):
    row_vals = {}
    for c in df.columns:
        if c == "_rescued_data":
            row_vals[c] = None
        elif c == id_col:
            row_vals[c] = existing_max_id + 1 + i  # unique primary key, no duplicates
        elif c == name_col:
            row_vals[c] = random_names[i] if i not in null_indices_name else None
        elif c == "product":
            row_vals[c] = random.choice(products) if i not in null_indices_product else None
        elif c == "remarks":
            if i in null_indices_remarks:
                row_vals[c] = None
            else:
                row_vals[c] = inject_special_chars(random.choice(remarks_vals))
        elif c == "order_status":
            row_vals[c] = random.choice(order_statuses) if i not in null_indices_order else None
        elif c in max_vals and max_vals[c] is not None:
            row_vals[c] = round(random.uniform(0, max_vals[c]), 2) if dict(df.dtypes)[c] in ("double", "float") else random.randint(0, int(max_vals[c]))
        else:
            row_vals[c] = None  # no data present -> null instead of blank
    random_rows.append(Row(**row_vals))

random_df = spark.createDataFrame(random_rows, schema=df.schema)
df_with_records = df.unionByName(random_df)
display(df_with_records)

In [0]:
display(df_with_records)

In [0]:
from pyspark.sql.functions import col, when, concat_ws, lit

quality_exprs = [when(col(c).isNull(), lit(f"{c} is missing")) for c in ["id", "customer_name"]]

quality_concat = concat_ws("|", *quality_exprs)

df_quality = df_with_records.withColumn(
    "quality check",
 when(quality_concat == "", lit("no errors")).otherwise(quality_concat),
)
display(df_quality)

In [0]:
from pyspark.sql.functions import when, col

df_dq = df_quality.withColumn(
    "dq_flag",
    when(col("quality check") == "no errors", "Y").otherwise("N"),
)
display(df_dq)

In [0]:
df_cleansed = df_dq.filter(col("dq_flag") == "Y")
display(df_cleansed)

df_bad_records = df_dq.filter(col("dq_flag") == "N")
display(df_bad_records)


In [0]:
from pyspark.sql.functions import col, initcap, regexp_replace, trim

# Start from the latest enriched dataframe
df_std = df_cleansed

# 1. customer_name -> initcap
df_std = df_std.withColumn("customer_name", initcap(col("customer_name").cast("string")))

# 2. product -> initcap
df_std = df_std.withColumn("product", initcap(col("product").cast("string")))

# 3. remarks -> keep only alphabets, numbers, and spaces; trim leading/trailing spaces but keep spaces in the middle
df_std = df_std.withColumn("remarks", trim(regexp_replace(col("remarks").cast("string"), r"[^A-Za-z0-9 ]", "")))

# 4. quantity -> replace null with 0
df_std = df_std.withColumn("quantity", when(col("quantity").isNull(), 0).otherwise(col("quantity")))

# 5. price -> keep only numbers and decimal point (strip $ and other symbols), then cast back to int
df_std = df_std.withColumn("price", regexp_replace(col("price").cast("string"), r"[^0-9.]", "").cast("int"))

display(df_std)

In [0]:
from pyspark.sql.functions import col, when, lit

df_std = df_std.withColumn(
    "product",
    when(col("product").isNull(), lit("guns")).otherwise(col("product")),
)
display(df_std)

In [0]:
from pyspark.sql.functions import col, when, lit

df_std = df_std.withColumn(
    "product",
    when(col("product").isNull() | (col("product") == "mouse") | (col("product") == "Mouse"), lit("guns")).otherwise(col("product")),
)
display(df_std)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank

wind_spec = Window.partitionBy("product", "order_status").orderBy(col("price").desc())
df_wind = df_std.withColumn("rank_num", dense_rank().over(wind_spec))
df_filter_1 = df_wind.filter(col("rank_num") == 1)
df_filter_2 = df_wind.filter(col("rank_num") != 1)
display(df_filter_1)
display(df_filter_2)

In [0]:
df_merged = df_filter_1.unionAll(df_filter_2)
display(df_merged )

In [0]:
from pyspark.sql.functions import sum as spark_sum, count

df_agg = df_std.groupBy("product").agg(spark_sum("price").alias("total_price"), count("*").alias("total_records"))
display(df_agg)

In [0]:
import math
from pyspark.sql.functions import col, count, lit, concat_ws, floor, rand, when

partition_col = "product"
salt_buckets = 10

# 1. Count records per partition key
partition_counts = (
    df_std.groupBy(partition_col)
    .agg(count("*").alias("record_count"))
    .orderBy(col("record_count").desc())
)
display(partition_counts)

# 2. IQR spike detection
counts_sorted = sorted(r["record_count"] for r in partition_counts.collect())

def percentile(vals, p):
    if not vals:
        return 0
    k = (len(vals) - 1) * p / 100.0
    f, c = math.floor(k), math.ceil(k)
    return vals[int(k)] if f == c else vals[f] * (c - k) + vals[c] * (k - f)

q1, q3 = percentile(counts_sorted, 25), percentile(counts_sorted, 75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print(f"Q1={q1}  Q3={q3}  IQR={iqr}  Upper fence={upper_fence}")

# 3. Flag spiked partitions
spiked_products = {
    r[partition_col] for r in partition_counts.collect()
    if r["record_count"] > upper_fence
}
print(f"Spiked partitions (count > {upper_fence}): {spiked_products}")

# 4. Salting — split each spiked partition into N sub-partitions
salt = floor(rand() * lit(salt_buckets)).cast("int")
df_salted = (
    df_std
    .withColumn(
        "salted_product",
        when(
            col(partition_col).isin(spiked_products),
            concat_ws("_", col(partition_col).cast("string"), salt),
        ).otherwise(col(partition_col).cast("string")),
    )
)


In [0]:
display(df_salted.groupBy("salted_product").agg(count("*").alias("record_count")).orderBy(col("record_count").desc()))

# 5. Write using the salted key as the partition column
# (df_salted.write.partitionBy("salted_product").mode("overwrite")
#     .saveAsTable("workspace.auto_loader.demo_table_cleansed"))

In [0]:
import math
from pyspark.sql.functions import col, count, lit, concat_ws, floor, rand, when, sum as spark_sum, length, coalesce

partition_col = "product"
target_partition_size_mb = 128  # target size per partition in MB; adjust as needed
salt_buckets = 10

# 1. Estimate the size (in bytes) of each partition key
#    Spark doesn't directly expose per-partition-byte size, so we approximate
#    by measuring the serialized size of all string/binary columns and
#    using a fixed overhead estimate per row for numeric columns.

# Collect the DataFrame to estimate in-memory size per row per partition
# Use spark's internal size estimator via RDD mapPartitions

from pyspark.sql.functions import sum as _spark_sum

# Build a size-estimation expression: sum of byte lengths of string columns
# plus a fixed overhead (8 bytes) for each numeric column
byte_size_exprs = []
for c, dtype in df_std.dtypes:
    if c == "_rescued_data":
        continue
    if dtype == "string":
        byte_size_exprs.append(coalesce(length(col(c).cast("string")).cast("long"), lit(0)))
    elif dtype in ("int", "bigint", "double", "float", "long", "short"):
        byte_size_exprs.append(lit(8).cast("long"))
    else:
        # Fallback: cast to string and measure length
        byte_size_exprs.append(coalesce(length(col(c).cast("string")).cast("long"), lit(0)))

row_size_expr = lit(0).cast("long")
for expr in byte_size_exprs:
    row_size_expr = row_size_expr + expr

print("=" * 70)
print("STEP 1: Estimate per-partition-key size (in bytes)")
print("=" * 70)
print(f"Partition column : {partition_col}")
print(f"Target size/part : {target_partition_size_mb} MB")
print(f"Max salt buckets  : {salt_buckets}")
print()

partition_sizes = (
    df_std
    .groupBy(partition_col)
    .agg(
        count("*").alias("record_count"),
        _spark_sum(row_size_expr).alias("estimated_size_bytes"),
    )
    .orderBy(col("estimated_size_bytes").desc())
)
print("Per-partition-key estimated sizes:")
print("-" * 70)
for r in partition_sizes.collect():
    mb = r["estimated_size_bytes"] / (1024 * 1024)
    print(f"  {r[partition_col]:>20s} | records: {r['record_count']:>8d} | size: {r['estimated_size_bytes']:>12d} bytes ({mb:.2f} MB)")
print()
display(partition_sizes)

# 2. IQR spike detection on sizes (in bytes)
sizes_sorted = sorted(r["estimated_size_bytes"] for r in partition_sizes.collect())

def percentile(vals, p):
    if not vals:
        return 0
    k = (len(vals) - 1) * p / 100.0
    f, c = math.floor(k), math.ceil(k)
    return vals[int(k)] if f == c else vals[f] * (c - k) + vals[c] * (k - f)

q1, q3 = percentile(sizes_sorted, 25), percentile(sizes_sorted, 75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

target_partition_size_bytes = target_partition_size_mb * 1024 * 1024

print("=" * 70)
print("STEP 2: IQR spike detection on partition sizes")
print("=" * 70)
print(f"  Q1 (25th percentile)   : {q1:,.0f} bytes ({q1 / (1024*1024):.2f} MB)")
print(f"  Q3 (75th percentile)   : {q3:,.0f} bytes ({q3 / (1024*1024):.2f} MB)")
print(f"  IQR                    : {iqr:,.0f} bytes ({iqr / (1024*1024):.2f} MB)")
print(f"  Upper fence (Q3+1.5*IQR): {upper_fence:,.0f} bytes ({upper_fence / (1024*1024):.2f} MB)")
print(f"  Target partition size  : {target_partition_size_mb} MB ({target_partition_size_bytes:,} bytes)")
print()

print("=" * 70)
print("STEP 3: Flag partitions that are skewed by SIZE")
print("=" * 70)
print(f"  A partition is spiked if its size exceeds BOTH:")
print(f"    - IQR upper fence : {upper_fence:,.0f} bytes ({upper_fence / (1024*1024):.2f} MB)")
print(f"    - Target size     : {target_partition_size_bytes:,} bytes ({target_partition_size_mb} MB)")
print()

spiked_products = {
    r[partition_col] for r in partition_sizes.collect()
    if r["estimated_size_bytes"] > upper_fence
    and r["estimated_size_bytes"] > target_partition_size_bytes
}

if spiked_products:
    print(f"  Spiked partitions found ({len(spiked_products)}):")
    for r in partition_sizes.collect():
        if r[partition_col] in spiked_products:
            mb = r["estimated_size_bytes"] / (1024 * 1024)
            print(f"    -> {r[partition_col]:>20s} | {r['estimated_size_bytes']:>12,} bytes ({mb:.2f} MB)")
else:
    print("  No spiked partitions found.")
print()

print("=" * 70)
print("STEP 4: Compute salt buckets per spiked partition")
print("=" * 70)
print(f"  Formula: buckets = ceil(estimated_size / target_size), capped at {salt_buckets}")
print()

partition_bucket_map = {}
for r in partition_sizes.collect():
    p = r[partition_col]
    if p in spiked_products:
        buckets = max(2, math.ceil(r["estimated_size_bytes"] / target_partition_size_bytes))
        partition_bucket_map[p] = min(buckets, salt_buckets)  # cap at salt_buckets
    else:
        partition_bucket_map[p] = 1

print(f"  {'Product':>20s} | {'Size (bytes)':>14s} | {'Size (MB)':>10s} | {'Buckets':>8s}")
print(f"  {'-'*20}-+-{'-'*14}-+-{'-'*10}-+-{'-'*8}")
for r in partition_sizes.collect():
    p = r[partition_col]
    b = partition_bucket_map[p]
    mb = r["estimated_size_bytes"] / (1024 * 1024)
    marker = " <-- spiked" if b > 1 else ""
    print(f"  {p:>20s} | {r['estimated_size_bytes']:>14,} | {mb:>10.2f} | {b:>8d}{marker}")
print()

print("=" * 70)
print("STEP 5: Salt spiked partitions into sub-partitions")
print("=" * 70)

# We use a broadcast mapping so each row picks a random bucket in [0, num_buckets)
from pyspark.sql.functions import rand as spark_rand, floor as spark_floor

# Create a small lookup DataFrame: product -> num_buckets
bucket_rows = [(k, v) for k, v in partition_bucket_map.items() if v > 1]
if bucket_rows:
    bucket_df = spark.createDataFrame(bucket_rows, [partition_col, "num_buckets"])

    # Join df_std with bucket_df to get num_buckets for spiked rows
    df_joined = df_std.join(bucket_df, on=partition_col, how="left")

    # Assign a random salt in [0, num_buckets) for spiked rows; 0 otherwise
    rand_salt = spark_floor(spark_rand() * col("num_buckets")).cast("int")
    df_salted = df_joined.withColumn(
        "salted_product",
        when(
            col("num_buckets").isNotNull(),
            concat_ws("_", col(partition_col).cast("string"), rand_salt.cast("string")),
        ).otherwise(col(partition_col).cast("string")),
    ).drop("num_buckets")
    print(f"  Applied salting to {len(bucket_rows)} spiked partition(s):")
    for k, v in bucket_rows:
        print(f"    -> {k} split into {v} sub-partitions (salted_product = '<product>_<0..{v-1}>')")
else:
    # No spiked partitions found — keep the original column as-is
    df_salted = df_std.withColumn("salted_product", col(partition_col).cast("string"))
    print("  No salting needed — keeping original partition column as-is.")
print()

print("=" * 70)
print("RESULT: Salted partition distribution")
print("=" * 70)
salted_summary = df_salted.groupBy("salted_product").agg(count("*").alias("record_count")).orderBy(col("record_count").desc())
for r in salted_summary.collect():
    print(f"  {r['salted_product']:>25s} | records: {r['record_count']:>8d}")
print()
print("Done. Use df_salted.write.partitionBy('salted_product') ... to write.")
print("=" * 70)
display(salted_summary)

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import random
import string

# ---------------------------------------------------------------------------
# Generate a messy dataset for data-cleaning practice
# Columns: id, name, phonenumber, adhar_number, address, remarks
# Each column has deliberate messy data:
#   • extra leading / trailing / embedded spaces
#   • special characters (@, #, $, %, &, *, !, ~, ^, etc.)
#   • mixed casing, inconsistent delimiters, nulls, duplicates
#   • phone & aadhaar numbers with random symbols / letters
# ---------------------------------------------------------------------------

random.seed(42)

special_chars = ["@", "#", "$", "%", "&", "*", "!", "~", "^", "+", "=", "`", "|"]

clean_names = [
    "Alice Johnson", "Bob Smith", "Charlie Brown", "Diana Prince", "Edward Norton",
    "Fiona Apple", "George Lucas", "Hannah Montana", "Ian McKellen", "Julia Roberts",
    "Kevin Hart", "Laura Linney", "Michael Caine", "Natalie Portman", "Oscar Wilde",
    "Penelope Cruz", "Quentin Tarantino", "Rachel Green", "Samuel Jackson", "Tina Fey",
    "Uma Thurman", "Vin Diesel", "Will Smith", "Xavier Dolan", "Yvonne Strahovski",
]

cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Mumbai", "Delhi", "Bangalore", "Pune", "Chennai"]
streets = ["MG Road", "Park Avenue", "Main Street", "Oak Lane", "Hill View", "Station Road"]

remarks_pool = ["good customer", "follow up needed", "vip client", "payment pending", "new lead", "escalation required"]

def mess_spaces(text):
    """Add random extra spaces at start, end, and between words."""
    words = str(text).split()
    # Random extra spaces between words
    messed = ("  " if random.random() < 0.5 else " ").join(words)
    # Random leading spaces
    if random.random() < 0.5:
        messed = " " * random.randint(1, 4) + messed
    # Random trailing spaces
    if random.random() < 0.5:
        messed = messed + " " * random.randint(1, 4)
    return messed

def inject_special(text):
    """Insert 1-3 random special characters at random positions."""
    chars = list(str(text))
    for _ in range(random.randint(1, 3)):
        pos = random.randint(0, len(chars))
        chars.insert(pos, random.choice(special_chars))
    return "".join(chars)

def mess_name(name):
    """Mess up a name: extra spaces, special chars, random casing, swapped word order."""
    n = mess_spaces(name)
    n = inject_special(n)
    if random.random() < 0.3:
        n = n.upper()
    elif random.random() < 0.3:
        n = n.lower()
    return n

def mess_phone():
    """Generate a messy phone number with symbols, letters, spaces, varying formats."""
    digits = "".join(random.choice("0123456789") for _ in range(10))
    style = random.choice(["separators", "mixed", "special", "letters"])
    if style == "separators":
        phone = f"{digits[:3]}-{digits[3:6]}-{digits[6:]}"
    elif style == "mixed":
        phone = f"+91 {digits[:5]} {digits[5:]}"
    elif style == "special":
        phone = inject_special(f"{digits[:3]}-{digits[3:6]}-{digits[6:]}")
    else:
        # Insert random letters
        phone = f"{digits[:5]}" + random.choice(string.ascii_letters) + f"{digits[5:]})"
        phone = inject_special(phone)
    # Occasionally add extra spaces
    if random.random() < 0.4:
        phone = mess_spaces(phone)
    return phone

def mess_aadhaar():
    """Generate a messy 12-digit Aadhaar number with spaces, dashes, letters, special chars."""
    digits = "".join(random.choice("0123456789") for _ in range(12))
    style = random.choice(["spaced", "dashed", "special", "letters", "plain"])
    if style == "spaced":
        aadhaar = f"{digits[:4]} {digits[4:8]} {digits[8:]}"
    elif style == "dashed":
        aadhaar = f"{digits[:4]}-{digits[4:8]}-{digits[8:]}"
    elif style == "special":
        aadhaar = inject_special(f"{digits[:4]}-{digits[4:8]}-{digits[8:]}")
    elif style == "letters":
        aadhaar = digits[:6] + random.choice(string.ascii_letters) + digits[6:]
        aadhaar = inject_special(aadhaar)
    else:
        aadhaar = digits
    if random.random() < 0.4:
        aadhaar = mess_spaces(aadhaar)
    return aadhaar

def mess_address():
    """Generate a messy address with extra spaces, special chars, mixed separators."""
    num = random.randint(1, 999)
    street = random.choice(streets)
    city = random.choice(cities)
    sep = random.choice([",", " ,", ", ", "  ,  ", " | ", " @ "])
    addr = f"{num} {street}{sep}{city}"
    addr = mess_spaces(addr)
    if random.random() < 0.5:
        addr = inject_special(addr)
    if random.random() < 0.3:
        addr = addr.upper()
    elif random.random() < 0.3:
        addr = addr.lower()
    return addr

def mess_remarks():
    """Generate messy remarks with extra spaces, special chars, inconsistent casing."""
    base = random.choice(remarks_pool)
    r = mess_spaces(base)
    r = inject_special(r)
    if random.random() < 0.4:
        r = r.upper()
    return r

rows = []
for i in range(1, 51):
    # ~15% of rows have null id or duplicate id for messy data
    row_id = existing_max_id if False else i  # placeholder; we'll override below
    pass

# Actually build the rows
rows = []
used_ids = set()
for i in range(1, 51):
    # id: mostly unique, but inject a few duplicates and some nulls
    if random.random() < 0.08:
        row_id = None
    elif random.random() < 0.1 and used_ids:
        row_id = random.choice(list(used_ids))  # duplicate id
    else:
        row_id = i
        used_ids.add(row_id)

    # name: ~10% null, otherwise messy
    name = None if random.random() < 0.1 else mess_name(random.choice(clean_names))

    # phonenumber: ~10% null
    phonenumber = None if random.random() < 0.1 else mess_phone()

    # adhar_number: ~12% null
    adhar_number = None if random.random() < 0.12 else mess_aadhaar()

    # address: ~8% null
    address = None if random.random() < 0.08 else mess_address()

    # remarks: ~15% null
    remarks = None if random.random() < 0.15 else mess_remarks()

    rows.append(Row(
        id=row_id,
        name=name,
        phonenumber=phonenumber,
        adhar_number=adhar_number,
        address=address,
        remarks=remarks,
    ))

# Define schema explicitly (id as int, rest as string)
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("phonenumber", StringType(), True),
    StructField("adhar_number", StringType(), True),
    StructField("address", StringType(), True),
    StructField("remarks", StringType(), True),
])

df_messy = spark.createDataFrame(rows, schema=schema)

print(f"Total rows: {df_messy.count()}")
print(f"Columns  : {df_messy.columns}")
print()
display(df_messy)

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, initcap

# 1. Clean name: remove special chars, trim extra spaces, title-case
df_clean = df_messy.withColumn("name", initcap(trim(regexp_replace(col("name"), r"[^a-zA-Z ]", ""))))

# 2. Clean phonenumber: keep only digits, then format as XXX-XXX-XXXX
df_clean = (
    df_clean
    .withColumn("phonenumber", regexp_replace(col("phonenumber").cast("string"), r"[^0-9]", ""))
    .withColumn("phonenumber", regexp_replace(col("phonenumber"), r"(\d{3})(\d{3})(\d{4})", r"$1-$2-$3"))
)
#clean adhar number 
df_clean = (
    df_clean
    .withColumn("adhar_number",regexp_replace(col("adhar_number"),r"[^0-9]",""))
    .withColumn("adhar_number",regexp_replace(col("adhar_number"),r"(\d{4})(\d{4})(\d{4})", r"$1-$2-$3")) 
)
# 4. Clean remarks: keep only alphanumerics and spaces, trim extra spaces
from pyspark.sql.functions import split, concat_ws, initcap as _initcap

df_clean = (
    df_clean
    .withColumn("remarks", trim(regexp_replace(col("remarks").cast("string"), r"[^A-Za-z0-9 ]", "")))
    .withColumn("remarks", regexp_replace(col("remarks"), r"\s+", " "))
    .withColumn("remarks", initcap(col("remarks")))
)

# 5. Clean address: keep only alphanumerics, spaces, commas, and hyphens; trim extra spaces, title-case

df_clean = (
    df_clean
    .withColumn("address", trim(regexp_replace(col("address").cast("string"), r"[^A-Za-z0-9 ,\-]", "")))
    .withColumn("address", regexp_replace(col("address"), r"\s+", " "))
    .withColumn("address", initcap(col("address")))
)

display(df_clean)

In [0]:
import random
from pyspark.sql.functions import col, regexp_replace, trim, initcap, when, lit
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

random.seed(99)

countries = [
    "United States", "India", "United Kingdom", "Canada", "Australia",
    "Germany", "France", "Singapore", "United Arab Emirates", "Japan",
]

currency_symbols = ["$", "€", "£", "₹", "A$", "C$"]

def mess_net_worth():
    """Generate a messy net-worth string with random currency symbol, commas, spaces, and special chars."""
    amount = random.randint(1000, 9_999_999)
    symbol = random.choice(currency_symbols)
    style = random.choice(["plain", "commas", "spaces", "special", "words"])
    if style == "plain":
        worth = f"{symbol}{amount}"
    elif style == "commas":
        worth = f"{symbol}{amount:,}"
    elif style == "spaces":
        worth = f"{symbol} {amount:,}"
    elif style == "special":
        chars = list(f"{symbol}{amount:,}")
        for _ in range(random.randint(1, 3)):
            chars.insert(random.randint(0, len(chars)), random.choice(["@", "#", "%", "&", "*"]))
        worth = "".join(chars)
    else:
        worth = f"{symbol}{amount:,} net worth"
    if random.random() < 0.3:
        worth = worth.upper()
    return worth

def mess_country():
    """Pick a country and mess it up with extra spaces, special chars, or random casing."""
    base = random.choice(countries)
    style = random.choice(["clean", "spaces", "special", "upper", "lower"])
    if style == "clean":
        return base
    elif style == "spaces":
        words = base.split()
        return ("  " if random.random() < 0.5 else " ").join(words)
    elif style == "special":
        chars = list(base)
        for _ in range(random.randint(1, 2)):
            chars.insert(random.randint(0, len(chars)), random.choice(["@", "#", "$", "%"]))
        return "".join(chars)
    elif style == "upper":
        return base.upper()
    else:
        return base.lower()

# Build enriched rows from df_clean, adding net_worth and country
existing_rows = df_clean.collect()
enriched_rows = []
for r in existing_rows:
    enriched_rows.append(Row(
        id=r["id"],
        name=r["name"],
        phonenumber=r["phonenumber"],
        adhar_number=r["adhar_number"],
        address=r["address"],
        remarks=r["remarks"],
        net_worth=None if random.random() < 0.10 else mess_net_worth(),
        country=None if random.random() < 0.08 else mess_country(),
    ))

enriched_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("phonenumber", StringType(), True),
    StructField("adhar_number", StringType(), True),
    StructField("address", StringType(), True),
    StructField("remarks", StringType(), True),
    StructField("net_worth", StringType(), True),
    StructField("country", StringType(), True),
])

df_enriched = spark.createDataFrame(enriched_rows, schema=enriched_schema)

# ---- Clean the new columns ----
# net_worth: strip currency symbols and special chars, keep digits and commas -> cast to double
from pyspark.sql.functions import regexp_replace as _rr
df_enriched = df_enriched.withColumn(
    "net_worth",
    _rr(col("net_worth").cast("string"), r"[^0-9]", "").cast("double"),
)

# country: remove special chars, trim, initcap
df_enriched = df_enriched.withColumn(
    "country",
    initcap(trim(regexp_replace(col("country").cast("string"), r"[^A-Za-z ]", ""))),
)

print(f"Total rows: {df_enriched.count()}")
print(f"Columns  : {df_enriched.columns}")
print()
display(df_enriched)

In [0]:
df_enriched.createOrReplaceTempView("df_enriched_tempview")



In [0]:
%sql
SELECT *, row_number() over(partition by country order by net_worth DESC) as row_num,
        sum(net_worth) over(partition by country order by net_worth DESC) as sum_by_country,
        sum(net_worth) over(partition by country order by net_worth DESC range between unbounded preceding and current row) as running_total,
        sum(net_worth) over(partition by country) as country_total_net_worth,
        round(avg(net_worth) over(partition by country),0) as country_total_avg_worth,
        round(try_subtract(net_worth,country_total_avg_worth),0) as subtract_amount,
        avg(net_worth) over(partition by country order by net_worth DESC) as country_avg_country_wise_net_worth
        FROM df_enriched_tempview